In [2]:
# ─── Cell 1: Load IPL data ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import os

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

BASE        = "/Users/aryanjungchhetri/Developers/6th sem/individual"
IPL_FOLDER  = f"{BASE}/data/raw/ipl_raw"

batting_df  = pd.read_csv(f"{BASE}/data/processed/ipl_batting_raw.csv")
bowling_df  = pd.read_csv(f"{BASE}/data/processed/ipl_bowling_raw.csv")

print(f"✅ IPL Batting : {batting_df.shape}")
print(f"✅ IPL Bowling : {bowling_df.shape}")


# ─── Cell 2: Batting position tracking ───────────────────────────────────────
all_files        = [f for f in os.listdir(IPL_FOLDER) if f.endswith(".json")]
position_records = []

for filename in all_files:
    with open(os.path.join(IPL_FOLDER, filename), "r") as f:
        match = json.load(f)

    info  = match["info"]
    teams = info["teams"]

    for inning in match["innings"]:
        batting_team  = inning["team"]
        batting_order = []

        for over_data in inning["overs"]:
            for delivery in over_data["deliveries"]:
                batter = delivery["batter"]
                if batter not in batting_order:
                    batting_order.append(batter)
                non_striker = delivery["non_striker"]
                if non_striker not in batting_order:
                    batting_order.append(non_striker)

        for position, player in enumerate(batting_order, start=1):
            position_records.append({
                "player"    : player,
                "match_file": filename,
                "team"      : batting_team,
                "position"  : position
            })

position_df = pd.DataFrame(position_records)
print(f"✅ IPL Position records: {position_df.shape}")
print(position_df.head(5).to_string())


# ─── Cell 3: Aggregate batting position ──────────────────────────────────────
position_agg = position_df.groupby("player")["position"].agg([
    "mean", "median", "min", "max",
    lambda x: x.mode()[0]
]).reset_index()

position_agg.columns = [
    "player", "avg_position", "median_position",
    "highest_position", "lowest_position", "typical_position"
]

def assign_position_role(pos):
    if pos <= 2:   return "opener"
    elif pos <= 4: return "top_order"
    elif pos <= 6: return "middle_order"
    elif pos <= 8: return "lower_order"
    else:          return "tailender"

position_agg["position_role"] = position_agg["typical_position"].apply(assign_position_role)

print("✅ IPL Position aggregated")
print("\n=== POSITION ROLE DISTRIBUTION ===")
print(position_agg["position_role"].value_counts().to_string())


# ─── Cell 4: IPL Batting features ────────────────────────────────────────────
def compute_batting_features(df):
    grp = pd.DataFrame()
    g   = df.groupby("player")

    # Basic
    grp["total_runs"]             = g["runs"].sum()
    grp["total_matches"]          = g["match_file"].nunique()
    grp["total_balls_faced"]      = g["balls_faced"].sum()
    grp["total_fours"]            = g["fours"].sum()
    grp["total_sixes"]            = g["sixes"].sum()
    grp["total_dismissed"]        = g["dismissed"].sum()
    grp["avg_runs"]               = g["runs"].mean().round(2)
    grp["highest_score"]          = g["runs"].max()
    grp["lowest_score"]           = g["runs"].min()

    # Not out & duck rate
    grp["not_out_rate"]           = (1 - (grp["total_dismissed"] / grp["total_matches"])).round(3)
    grp["duck_count"]             = g.apply(lambda x: ((x["runs"] == 0) & (x["dismissed"] == 1)).sum())
    grp["duck_rate"]              = (grp["duck_count"] / grp["total_matches"]).round(3)

    # Consistency
    grp["consistency_score"]      = g["runs"].std().fillna(0).round(2)
    grp["innings_above_30"]       = g.apply(lambda x: (x["runs"] >= 30).sum())
    grp["innings_above_50"]       = g.apply(lambda x: (x["runs"] >= 50).sum())
    grp["innings_above_30_pct"]   = (grp["innings_above_30"] / grp["total_matches"]).round(3)
    grp["innings_above_50_pct"]   = (grp["innings_above_50"] / grp["total_matches"]).round(3)

    # Strike rate & boundary
    grp["strike_rate"]            = (grp["total_runs"] / grp["total_balls_faced"].replace(0, np.nan) * 100).round(2)
    grp["boundary_rate"]          = ((grp["total_fours"] + grp["total_sixes"]) / grp["total_balls_faced"].replace(0, np.nan)).round(3)
    grp["six_rate"]               = (grp["total_sixes"] / grp["total_balls_faced"].replace(0, np.nan)).round(3)
    grp["four_rate"]              = (grp["total_fours"] / grp["total_balls_faced"].replace(0, np.nan)).round(3)
    boundary_runs                 = (grp["total_fours"] * 4) + (grp["total_sixes"] * 6)
    grp["boundary_runs_pct"]      = (boundary_runs / grp["total_runs"].replace(0, np.nan)).round(3)
    grp["dot_ball_rate"]          = (g["dot_balls"].sum() / grp["total_balls_faced"].replace(0, np.nan)).round(3)

    # Phase runs %
    pp_runs   = g["pp_runs"].sum()
    mid_runs  = g["mid_runs"].sum()
    dth_runs  = g["death_runs"].sum()
    pp_balls  = g["pp_balls"].sum()
    mid_balls = g["mid_balls"].sum()
    dth_balls = g["death_balls"].sum()

    grp["pp_runs_pct"]            = (pp_runs  / grp["total_runs"].replace(0, np.nan)).round(3)
    grp["mid_runs_pct"]           = (mid_runs / grp["total_runs"].replace(0, np.nan)).round(3)
    grp["death_runs_pct"]         = (dth_runs / grp["total_runs"].replace(0, np.nan)).round(3)

    # Phase strike rates
    grp["pp_strike_rate"]         = (pp_runs  / pp_balls.replace(0, np.nan)  * 100).round(2)
    grp["mid_strike_rate"]        = (mid_runs / mid_balls.replace(0, np.nan) * 100).round(2)
    grp["death_strike_rate"]      = (dth_runs / dth_balls.replace(0, np.nan) * 100).round(2)

    # Dominant phase
    phase_df                      = pd.DataFrame({
        "pp"   : grp["pp_strike_rate"].fillna(0),
        "mid"  : grp["mid_strike_rate"].fillna(0),
        "death": grp["death_strike_rate"].fillna(0)
    })
    grp["dominant_phase"]         = phase_df.idxmax(axis=1)

    # Win contribution
    win_mask                      = df["team"] == df["winner"]
    loss_mask                     = df["team"] != df["winner"]
    grp["avg_runs_in_wins"]       = df[win_mask].groupby("player")["runs"].mean().round(2)
    grp["avg_runs_in_losses"]     = df[loss_mask].groupby("player")["runs"].mean().round(2)

    # Team
    grp["team"]                   = g["team"].agg(lambda x: x.value_counts().index[0])

    return grp.reset_index()

batting_features = compute_batting_features(batting_df)
print(f"✅ IPL Batting features: {batting_features.shape}")



# ─── Cell 5: IPL Bowling features ────────────────────────────────────────────
def compute_bowling_features(df):
    grp = pd.DataFrame()
    g   = df.groupby("player")

    grp["total_wickets"]              = g["wickets"].sum()
    grp["total_matches_bowled"]       = g["match_file"].nunique()
    grp["total_balls_bowled"]         = g["balls_bowled"].sum()
    grp["total_runs_conceded"]        = g["runs_conceded"].sum()
    grp["total_wides"]                = g["wides"].sum()
    grp["total_noballs"]              = g["noballs"].sum()

    grp["economy_rate"]               = (grp["total_runs_conceded"] / (grp["total_balls_bowled"].replace(0, np.nan) / 6)).round(2)
    grp["bowling_average"]            = (grp["total_runs_conceded"] / grp["total_wickets"].replace(0, np.nan)).round(2)
    grp["bowling_strike_rate"]        = (grp["total_balls_bowled"] / grp["total_wickets"].replace(0, np.nan)).round(2)
    grp["avg_wickets_per_match"]      = (grp["total_wickets"] / grp["total_matches_bowled"]).round(3)
    grp["bowling_consistency"]        = g["wickets"].std().fillna(0).round(2)
    grp["two_plus_hauls"]             = g.apply(lambda x: (x["wickets"] >= 2).sum())
    grp["two_plus_haul_rate"]         = (grp["two_plus_hauls"] / grp["total_matches_bowled"]).round(3)
    grp["bowling_dot_rate"]           = (g["dot_balls"].sum() / grp["total_balls_bowled"].replace(0, np.nan)).round(3)
    grp["wide_rate"]                  = (grp["total_wides"] / grp["total_matches_bowled"]).round(3)
    grp["noball_rate"]                = (grp["total_noballs"] / grp["total_matches_bowled"]).round(3)

    pp_runs   = g["pp_runs"].sum()
    mid_runs  = g["mid_runs"].sum()
    dth_runs  = g["death_runs"].sum()
    pp_balls  = g["pp_balls"].sum()
    mid_balls = g["mid_balls"].sum()
    dth_balls = g["death_balls"].sum()

    grp["pp_economy"]                 = (pp_runs  / pp_balls.replace(0, np.nan)  * 6).round(2)
    grp["mid_economy"]                = (mid_runs / mid_balls.replace(0, np.nan) * 6).round(2)
    grp["death_economy"]              = (dth_runs / dth_balls.replace(0, np.nan) * 6).round(2)
    grp["pp_wicket_contribution"]     = (pp_balls  / grp["total_balls_bowled"].replace(0, np.nan)).round(3)
    grp["mid_wicket_contribution"]    = (mid_balls / grp["total_balls_bowled"].replace(0, np.nan)).round(3)
    grp["death_wicket_contribution"]  = (dth_balls / grp["total_balls_bowled"].replace(0, np.nan)).round(3)

    phase_eco                         = pd.DataFrame({
        "powerplay": grp["pp_economy"].fillna(999),
        "middle"   : grp["mid_economy"].fillna(999),
        "death"    : grp["death_economy"].fillna(999)
    })
    grp["dominant_phase"]             = phase_eco.idxmin(axis=1)

    def assign_bowling_role(row):
        if row["pp_wicket_contribution"]    >= 0.4: return "powerplay_bowler"
        elif row["death_wicket_contribution"] >= 0.4: return "death_bowler"
        elif row["mid_wicket_contribution"]   >= 0.5: return "middle_overs_bowler"
        else:                                         return "all_phases_bowler"

    grp["bowling_role"]               = grp.apply(assign_bowling_role, axis=1)

    win_mask                          = df["team"] == df["winner"]
    loss_mask                         = df["team"] != df["winner"]
    grp["wickets_in_wins"]            = df[win_mask].groupby("player")["wickets"].mean().round(2)
    grp["wickets_in_losses"]          = df[loss_mask].groupby("player")["wickets"].mean().round(2)
    grp["team"]                       = g["team"].agg(lambda x: x.value_counts().index[0])

    return grp.reset_index()

bowling_features = compute_bowling_features(bowling_df)
print(f"✅ IPL Bowling features: {bowling_features.shape}")


# ─── Cell 6: Merge position + assign batting role ─────────────────────────────
batting_features = batting_features.merge(
    position_agg[["player", "typical_position", "avg_position", "position_role"]],
    on="player", how="left"
)
batting_features["typical_position"] = batting_features["typical_position"].fillna(0)
batting_features["position_role"]    = batting_features["position_role"].fillna("unknown")

def assign_batting_role(row):
    pos   = row["typical_position"]
    pp_sr = row["pp_strike_rate"]    if not pd.isna(row["pp_strike_rate"])    else 0
    dt_sr = row["death_strike_rate"] if not pd.isna(row["death_strike_rate"]) else 0
    sr    = row["strike_rate"]       if not pd.isna(row["strike_rate"])       else 0
    avg   = row["avg_runs"]

    if pos <= 2 and pp_sr >= 120:   return "aggressive_opener"
    elif pos <= 2:                   return "anchor_opener"
    elif pos <= 4 and avg >= 25:     return "top_order_anchor"
    elif pos <= 4 and sr >= 140:     return "top_order_aggressor"
    elif pos <= 6 and dt_sr >= 150:  return "finisher"
    elif pos <= 6:                   return "middle_order"
    elif sr >= 150:                  return "lower_order_hitter"
    else:                            return "tailender"

batting_features["batting_role"] = batting_features.apply(assign_batting_role, axis=1)
print("✅ IPL batting role assigned")
print("\n=== IPL BATTING ROLE DISTRIBUTION ===")
print(batting_features["batting_role"].value_counts().to_string())



# ─── Cell 7: Filter & save ────────────────────────────────────────────────────
MIN_BAT_MATCHES  = 5   # Higher threshold for IPL (more matches available)
MIN_BOWL_MATCHES = 5

batting_final = batting_features[batting_features["total_matches"] >= MIN_BAT_MATCHES].copy()
bowling_final = bowling_features[bowling_features["total_matches_bowled"] >= MIN_BOWL_MATCHES].copy()

print(f"✅ IPL Batting : {len(batting_final)} players (was {len(batting_features)})")
print(f"✅ IPL Bowling : {len(bowling_final)} players (was {len(bowling_features)})")

batting_final.to_csv(f"{BASE}/data/processed/ipl_batting_features.csv", index=False)
bowling_final.to_csv(f"{BASE}/data/processed/ipl_bowling_features.csv", index=False)

print("\n✅ Saved:")
print("   → data/processed/ipl_batting_features.csv")
print("   → data/processed/ipl_bowling_features.csv")



# ─── Cell 8: Sanity checks ────────────────────────────────────────────────────
print("=== TOP 10 IPL BATTERS BY AVG RUNS ===")
print(batting_final.nlargest(10, "avg_runs")[
    ["player", "total_matches", "avg_runs", "strike_rate",
     "batting_role", "dominant_phase", "typical_position"]
].to_string())

print("\n=== TOP 10 IPL BOWLERS BY ECONOMY ===")
print(bowling_final.nsmallest(10, "economy_rate")[
    ["player", "total_matches_bowled", "economy_rate",
     "total_wickets", "bowling_role", "dominant_phase"]
].to_string())

print("\n=== TOP 10 IPL AGGRESSIVE OPENERS ===")
openers = batting_final[batting_final["batting_role"] == "aggressive_opener"]
print(f"Total: {len(openers)}")
print(openers.nlargest(10, "pp_strike_rate")[
    ["player", "total_matches", "avg_runs",
     "pp_strike_rate", "strike_rate", "boundary_rate"]
].to_string())

print("\n=== TOP 10 IPL FINISHERS ===")
finishers = batting_final[batting_final["batting_role"] == "finisher"]
print(f"Total: {len(finishers)}")
print(finishers.nlargest(10, "death_strike_rate")[
    ["player", "total_matches", "avg_runs",
     "death_strike_rate", "six_rate"]
].to_string())

print("\n=== TOP 10 IPL DEATH BOWLERS ===")
death = bowling_final[bowling_final["bowling_role"] == "death_bowler"]
print(f"Total: {len(death)}")
print(death.nsmallest(10, "death_economy")[
    ["player", "total_matches_bowled", "death_economy",
     "total_wickets", "two_plus_haul_rate"]
].to_string())

print("\n=== TOP 10 IPL POWERPLAY BOWLERS ===")
pp = bowling_final[bowling_final["bowling_role"] == "powerplay_bowler"]
print(f"Total: {len(pp)}")
print(pp.nsmallest(10, "pp_economy")[
    ["player", "total_matches_bowled", "pp_economy",
     "total_wickets", "bowling_dot_rate"]
].to_string())



✅ IPL Batting : (18057, 23)
✅ IPL Bowling : (14118, 22)
✅ IPL Position records: (18421, 4)
                player    match_file                 team  position
0              TM Head  1426261.json  Sunrisers Hyderabad         1
1      Abhishek Sharma  1426261.json  Sunrisers Hyderabad         2
2           AK Markram  1426261.json  Sunrisers Hyderabad         3
3  Nithish Kumar Reddy  1426261.json  Sunrisers Hyderabad         4
4          RA Tripathi  1426261.json  Sunrisers Hyderabad         5
✅ IPL Position aggregated

=== POSITION ROLE DISTRIBUTION ===
position_role
tailender       204
lower_order     159
opener          137
middle_order    119
top_order       105
✅ IPL Batting features: (720, 34)
✅ IPL Bowling features: (564, 28)
✅ IPL batting role assigned

=== IPL BATTING ROLE DISTRIBUTION ===
batting_role
tailender              325
middle_order           100
finisher                84
anchor_opener           71
aggressive_opener       66
lower_order_hitter      34
top_order_ancho